# Profiling & Benchmarks

Instrument forward passes with `Profiler`/`Timer`, track memory with
`MemoryTracker`, and run regression benchmarks with `sneppx-bench`.

In [ ]:
import time, numpy as np
from SneppX_ALG import (
    Transformer, Tensor, Profiler, Timer, MemoryTracker,
    get_profiler, timeit,
)

model = Transformer(vocab_size=500, dim=256, num_heads=4, num_layers=4,
                    ffn_dim=1024, max_seq_len=64)
x = Tensor.randn((8, 64))
print('ready')

## 1. Profile named regions

In [ ]:
prof = Profiler(enabled=True)
with Timer(prof, 'forward'):
    out = model(x)
with Timer(prof, 'loss'):
    loss = out.data.mean()
prof.print_summary()

## 2. Global profiler + @timeit

In [ ]:
g = get_profiler(); g.enabled = True; g.reset()

@timeit(g)
def step(x):
    return model(x)

for _ in range(5):
    step(x)
print(g.to_json()[:200])

## 3. Memory tracking

In [ ]:
mt = MemoryTracker()
mt.start()
for _ in range(10):
    _ = model(x)
peak = mt.peak(); mt.checkpoint('after-10'); mt.stop()
print('peak bytes:', peak)

## 4. CPU timing fallback (no C backend)

In [ ]:
t0 = time.perf_counter()
for _ in range(20):
    _ = model(x)
dt = (time.perf_counter() - t0) / 20
print(f'fwd {dt*1000:.2f} ms')

## 5. Regression benchmark (shell)

In [ ]:
# From a terminal:
#   sneppx-bench run tests/python/test_tensor.py --repeat 10 \
#       --save results/bench_$(git rev-parse --short HEAD).json
#   sneppx-bench compare results/bench_a.json results/bench_b.json
print('see sneppx-bench --help for flags')